## Options

Every setting is in the next cell. Change it there, then Run all.

In [ ]:
# --- Run folder ---
REUSE_RUN =  None       # None = start a new run_N folder, or e.g. 3 to work in run_3 (e.g. Vivado on old results)

# --- Which models to train (True = train, False = skip) ---
RUN = {'mnist_sm': True, 'mnist_lg': True, 'jsc_sm10': True, 'jsc_sm50': True, 'jsc_md': True, 'jsc_lg': True}

# --- Model settings (paper Table 14) and paper results (Table 2: accuracy %, LUTs, FFs) ---
CONFIGS = {
    'mnist_sm': dict(data='mnist', z=1,   layers=[1000, 500], tau=1/0.245, bs=128, epochs=100, step=30, paper=(97.1, 692, 422)),
    'mnist_lg': dict(data='mnist', z=3,   layers=[2000, 1000], tau=1/0.173, bs=128, epochs=100, step=30, paper=(98.3, 4082, 3385)),
    'jsc_sm10': dict(data='jsc',   z=200, layers=[10],   tau=1/0.7,  bs=100, epochs=32, step=14, paper=(71.1, 20, 22)),
    'jsc_sm50': dict(data='jsc',   z=200, layers=[50],   tau=1/0.3,  bs=100, epochs=32, step=14, paper=(74.0, 110, 72)),
    'jsc_md':   dict(data='jsc',   z=200, layers=[360],  tau=1/0.1,  bs=100, epochs=32, step=14, paper=(75.6, 720, 457)),
    'jsc_lg':   dict(data='jsc',   z=200, layers=[2400], tau=1/0.03, bs=100, epochs=32, step=14, paper=(76.3, 4972, 3305)),
}

# --- Verilog ---
INPUT_REG = False         # True adds a register on every input bit (not counted in the paper's Table 2)
FAST_RTL = True           # pipelined popcount + argmax tree; False = simple single-cycle version

# --- Vivado ---
VIVADO_PART = 'xc7a200tffg1156-2'
VIVADO_PERIOD = 4.0               # ns (250 MHz), for timing / Fmax
VIVADO_IMPL = True                # False = synthesis only
# Synthesis strategy, same names as the Vivado GUI. The paper used 'Flow_PerfOptimized_high'.
#   'Vivado Synthesis Defaults' | 'Flow_PerfOptimized_high' | 'Flow_AreaOptimized_high' | 'Flow_AreaOptimized_medium' | 'Flow_RuntimeOptimized'
VIVADO_SYNTH_STRATEGY = 'Flow_PerfOptimized_high'
# Implementation (place & route) strategy: 'Vivado Implementation Defaults' | 'Performance_Explore'
VIVADO_IMPL_STRATEGY = 'Vivado Implementation Defaults'
VIVADO_MODELS = None              # None = every .v in OUT, or e.g. ['jsc_sm10']
VIVADO_PATH = 'D:/2026.1/Vivado/bin/vivado.bat'   # None = search the usual install folders
VIVADO_SRC = None                 # folder with the .v files; None = this run's folder


SyntaxError: invalid syntax (3366055274.py, line 2)

## Setup

Install everything once from a terminal in this folder (the one with this notebook):

```
pip install -r requirements.txt
```



In [33]:
import os, sys, math, re, collections, pickle, time, subprocess, shutil, json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.functional import cross_entropy
from torchvision import datasets
import torch_dwn as dwn

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('device:', device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

BASE = os.path.abspath('.').replace('\\', '/')
_runs = [int(m.group(1)) for d in os.listdir(BASE) if os.path.isdir(os.path.join(BASE, d)) and (m := re.fullmatch(r'run_(\d+)', d))]
RUN_ID = REUSE_RUN if REUSE_RUN is not None else max(_runs, default=0) + 1
OUT = f'{BASE}/run_{RUN_ID}'
if REUSE_RUN is not None and not os.path.isdir(OUT):
    raise FileNotFoundError(f'{OUT} does not exist; existing runs: {sorted(_runs)}')
os.makedirs(OUT, exist_ok=True)
print(f'run #{RUN_ID}: results in', OUT)


device: cuda:0 NVIDIA GeForce GTX 1660


FileNotFoundError: c:/Users/youse/Downloads/DWN/run_1 does not exist; existing runs: []

## Data

In [ ]:
from sklearn.model_selection import train_test_split
RAW = {}
if any(RUN[k] and CONFIGS[k]['data'] == 'mnist' for k in CONFIGS):
    tr = datasets.MNIST('./data', train=True, download=True)
    te = datasets.MNIST('./data', train=False, download=True)
    RAW['mnist'] = (tr.data.reshape(-1, 784).float() / 255, tr.targets.clone(),
                    te.data.reshape(-1, 784).float() / 255, te.targets.clone(), 10)
if any(RUN[k] and CONFIGS[k]['data'] == 'jsc' for k in CONFIGS):
    import openml
    ds = openml.datasets.get_dataset(42468)
    df_x, df_y, _, _ = ds.get_data(dataset_format='dataframe', target=ds.default_target_attribute)
    features = df_x.values.astype(np.float32)
    names = list(df_y.unique())
    labels = np.array(df_y.map(lambda v: names.index(v)).values)
    xtr, xte, ytr, yte = train_test_split(features, labels, train_size=0.8, random_state=42)   # split used by LogicNets
    RAW['jsc'] = (torch.tensor(xtr), torch.tensor(ytr, dtype=torch.int64),
                  torch.tensor(xte), torch.tensor(yte, dtype=torch.int64), int(labels.max() + 1))
for k, v in RAW.items():
    print(k, 'train', tuple(v[0].shape), 'test', tuple(v[2].shape), 'classes', v[4])

mnist train (60000, 784) test (10000, 784) classes 10


## Training

In [ ]:
def evaluate(model, x_test, y_test, batch_size=4096):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, x_test.shape[0], batch_size):
            preds.append(model(x_test[i:i+batch_size].to(device)).argmax(dim=1).cpu())
    return (torch.cat(preds) == y_test).float().mean().item()

def train_and_evaluate(model, optimizer, scheduler, x_train, y_train, x_test, y_test, epochs, batch_size):
    n_samples = x_train.shape[0]
    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(n_samples)
        correct_train, t0 = 0, time.time()
        for i in range(0, n_samples, batch_size):
            optimizer.zero_grad()
            indices = permutation[i:i+batch_size]
            batch_x, batch_y = x_train[indices].to(device), y_train[indices].to(device)
            outputs = model(batch_x)
            loss = cross_entropy(outputs, batch_y)
            loss.backward()
            optimizer.step()
            correct_train += (outputs.argmax(dim=1) == batch_y).sum().item()
        scheduler.step()
        test_acc = evaluate(model, x_test, y_test)
        print(f'Epoch {epoch + 1}/{epochs}, Train Loss: {loss.item():.4f}, Train Accuracy: {correct_train / n_samples:.4f}, '
              f'Test Accuracy: {test_acc:.4f} ({time.time() - t0:.0f}s)')
    return test_acc

## Hardware: export, table-lookup simulation, Verilog

In [ ]:
def export_torch_dwn(model):
    # Wiring (luts, n) and boolean truth tables (luts, 2^n) from a trained torch_dwn model
    layers = [m for m in model if isinstance(m, dwn.LUTLayer)]
    gs = [m for m in model if isinstance(m, dwn.GroupSum)][0]
    spec = {'k': gs.k, 'layers': []}
    for l in layers:
        if isinstance(l.mapping, dwn.LearnableMapping):
            m = l.mapping.weights.argmax(dim=0).reshape(l.output_size, l.n)
        else:
            m = l.mapping
        spec['layers'].append({'mapping': m.detach().cpu().numpy().astype(np.int32),
                               'table': l.luts.detach().cpu().numpy() > 0})
    return spec

def simulate(spec, x_bits):
    # Pure table lookups and popcount, no floats
    h = np.asarray(x_bits, dtype=np.uint8)
    for layer in spec['layers']:
        m, t = layer['mapping'], layer['table']
        addr = np.zeros((h.shape[0], m.shape[0]), dtype=np.int64)
        for l in range(m.shape[1]):
            addr |= h[:, m[:, l]].astype(np.int64) << l
        h = t[np.arange(m.shape[0])[None, :], addr].astype(np.uint8)
    k = spec['k']
    h = np.pad(h, ((0, 0), (0, (-h.shape[1]) % k)))
    return h.reshape(h.shape[0], k, -1).sum(-1).argmax(1)

def lut_layer_assigns(m, t, src, dst):
    out = []
    J, n = m.shape
    for j in range(J):
        init = ''.join('1' if b else '0' for b in t[j][::-1])          # leftmost char = address 2^n-1
        sel = ', '.join(f'{src}[{int(i)}]' for i in m[j][::-1])        # address MSB first
        out.append(f"  assign {dst}[{j}] = {2**n}'b{init} >> {{{sel}}};")
    return out

def gen_pipelined(spec, n_inputs, name='dwn_top', input_reg=False):
    # [optional input reg] -> [LUT layer -> reg] x L -> popcount per class -> reg -> argmax -> output reg
    K = spec['k']
    Jl = spec['layers'][-1]['mapping'].shape[0]
    G = math.ceil(Jl / K)                       # bits per class
    W = max(1, math.ceil(math.log2(G + 1)))     # popcount width
    CW = max(1, math.ceil(math.log2(K)))        # class index width
    L = [f'module {name}(input clk, input [{n_inputs-1}:0] x, output reg [{CW-1}:0] y);']
    if input_reg:
        L += [f'  reg [{n_inputs-1}:0] x_r;', '  always @(posedge clk) x_r <= x;']
        prev = 'x_r'
    else:
        prev = 'x'
    for li, layer in enumerate(spec['layers']):
        J = layer['mapping'].shape[0]
        L += [f'  wire [{J-1}:0] h{li};', f'  reg  [{J-1}:0] h{li}_r;']
        L += lut_layer_assigns(layer['mapping'], layer['table'], prev, f'h{li}')
        L += [f'  always @(posedge clk) h{li}_r <= h{li};']
        prev = f'h{li}_r'
    padded = K * G
    L += [f"  wire [{padded-1}:0] f = {{{padded - Jl}'b0, {prev}}};" if padded > Jl else f'  wire [{padded-1}:0] f = {prev};',
          f'  reg [{W-1}:0] cnt [0:{K-1}];',
          '  integer c, i;',
          f'  reg [{W-1}:0] acc;',
          '  always @(posedge clk) begin',
          f'    for (c = 0; c < {K}; c = c + 1) begin',
          '      acc = 0;',
          f'      for (i = 0; i < {G}; i = i + 1) acc = acc + f[c*{G} + i];',
          '      cnt[c] <= acc;',
          '    end',
          '  end',
          f'  reg [{W-1}:0] best; reg [{CW-1}:0] best_i;',
          '  always @(*) begin',
          '    best = cnt[0]; best_i = 0;',
          f'    for (c = 1; c < {K}; c = c + 1) if (cnt[c] > best) begin best = cnt[c]; best_i = c; end',
          '  end',
          '  always @(posedge clk) y <= best_i;',
          'endmodule']
    return '\n'.join(L)

def gen_pipelined_fast(spec, n_inputs, name='dwn_top', input_reg=False, pc_group=25):
    # [optional input reg] -> [LUT layer -> reg] x L -> partial popcounts (groups of pc_group bits) -> reg
    # -> per-class sum -> reg -> argmax comparison tree, one register per level -> y
    K = spec['k']
    Jl = spec['layers'][-1]['mapping'].shape[0]
    G = math.ceil(Jl / K)
    W = max(1, math.ceil(math.log2(G + 1)))
    CW = max(1, math.ceil(math.log2(K)))
    L = [f'module {name}(input clk, input [{n_inputs-1}:0] x, output reg [{CW-1}:0] y);']
    if input_reg:
        L += [f'  reg [{n_inputs-1}:0] x_r;', '  always @(posedge clk) x_r <= x;']
        prev = 'x_r'
    else:
        prev = 'x'
    latency = 1 if input_reg else 0
    for li, layer in enumerate(spec['layers']):
        J = layer['mapping'].shape[0]
        L += [f'  wire [{J-1}:0] h{li};', f'  reg  [{J-1}:0] h{li}_r;']
        L += lut_layer_assigns(layer['mapping'], layer['table'], prev, f'h{li}')
        L += [f'  always @(posedge clk) h{li}_r <= h{li};']
        prev = f'h{li}_r'
        latency += 1
    padded = K * G
    L += [f"  wire [{padded-1}:0] f = {{{padded - Jl}'b0, {prev}}};" if padded > Jl else f'  wire [{padded-1}:0] f = {prev};']
    # popcount stage 1: partial sums
    for c in range(K):
        parts = []
        for q, s in enumerate(range(0, G, pc_group)):
            bits = list(range(c * G + s, c * G + min(s + pc_group, G)))
            pw = max(1, math.ceil(math.log2(len(bits) + 1)))
            L += [f'  reg [{pw-1}:0] p{c}_{q};',
                  f"  always @(posedge clk) p{c}_{q} <= {' + '.join(f'f[{b}]' for b in bits)};"]
            parts.append(f'p{c}_{q}')
        # popcount stage 2: per-class sum
        L += [f'  reg [{W-1}:0] cnt{c};', f"  always @(posedge clk) cnt{c} <= {' + '.join(parts)};"]
    latency += 2
    # argmax tree: left wins ties, so the lowest index wins like numpy/torch argmax
    nodes = [(f'cnt{c}', f"{CW}'d{c}") for c in range(K)]
    lvl = 0
    while len(nodes) > 2:
        nxt = []
        for j in range(0, len(nodes), 2):
            v, i = f'v{lvl}_{j//2}', f'i{lvl}_{j//2}'
            L += [f'  reg [{W-1}:0] {v}; reg [{CW-1}:0] {i};']
            if j + 1 < len(nodes):
                (va, ia), (vb, ib) = nodes[j], nodes[j + 1]
                L += [f'  always @(posedge clk) if ({va} >= {vb}) begin {v} <= {va}; {i} <= {ia}; end else begin {v} <= {vb}; {i} <= {ib}; end']
            else:
                va, ia = nodes[j]
                L += [f'  always @(posedge clk) begin {v} <= {va}; {i} <= {ia}; end']
            nxt.append((v, i))
        nodes = nxt
        lvl += 1
        latency += 1
    if len(nodes) == 2:
        (va, ia), (vb, ib) = nodes
        L += [f'  always @(posedge clk) y <= ({va} >= {vb}) ? {ia} : {ib};']
    else:
        L += [f'  always @(posedge clk) y <= {nodes[0][1]};']
    latency += 1
    L += ['endmodule']
    return '\n'.join(L), latency

def hardware_report(model, x_test, y_test, tag, input_reg=False, fast=True):
    spec = export_torch_dwn(model)
    model.eval()
    with torch.no_grad():
        torch_pred = torch.cat([model(x_test[i:i+4096].to(device)).argmax(1).cpu() for i in range(0, len(x_test), 4096)]).numpy()
    sim_pred = simulate(spec, x_test.numpy())
    y = np.asarray(y_test)
    r = {'model': tag, 'acc_torch': (torch_pred == y).mean(), 'acc_sim': (sim_pred == y).mean(),
         'agreement': (sim_pred == torch_pred).mean(), 'lut_nodes': sum(l['mapping'].shape[0] for l in spec['layers'])}
    print(f"[{tag}] acc torch {r['acc_torch']:.4f} | acc table-sim {r['acc_sim']:.4f} | agreement {r['agreement']:.4f}")
    pickle.dump(spec, open(f'{OUT}/{tag}_spec.pkl', 'wb'))
    v_path = f'{OUT}/{tag}.v'
    if fast:   # pipelined popcount + argmax tree (needed for high clock rates)
        v, r['latency_cycles'] = gen_pipelined_fast(spec, x_test.shape[1], input_reg=input_reg)
    else:      # simple version: one-cycle popcount, chained argmax
        v = gen_pipelined(spec, x_test.shape[1], input_reg=input_reg)
    open(v_path, 'w').write(v)
    return r

## Train, simulate and write Verilog for every selected model


In [ ]:
results, models = [], {}
json.dump({'run': RUN_ID, 'started': time.strftime('%Y-%m-%d %H:%M:%S'), 'RUN': RUN, 'INPUT_REG': INPUT_REG, 'FAST_RTL': FAST_RTL,
           'CONFIGS': {k: {kk: vv for kk, vv in v.items()} for k, v in CONFIGS.items() if RUN[k]}},
          open(f'{OUT}/run_info.json', 'w'), indent=1, default=str)
for tag, cfg in CONFIGS.items():
    if not RUN[tag]:
        continue
    print(f'\n===== {tag} =====')
    x_train, y_train, x_test, y_test, k = RAW[cfg['data']]
    therm = dwn.DistributiveThermometer(cfg['z']).fit(x_train)
    xb_train = therm.binarize(x_train).flatten(start_dim=1)
    xb_test = therm.binarize(x_test).flatten(start_dim=1)

    layers, width = [], xb_train.shape[1]
    for i, h in enumerate(cfg['layers']):
        layers.append(dwn.LUTLayer(width, h, n=6, mapping='learnable' if i == 0 else 'random'))
        width = h
    model = nn.Sequential(*layers, dwn.GroupSum(k=k, tau=cfg['tau'])).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, gamma=0.1, step_size=cfg['step'])
    train_and_evaluate(model, optimizer, scheduler, xb_train, y_train, xb_test, y_test, cfg['epochs'], cfg['bs'])
    # save the trained model so it can be reused without retraining: the whole PyTorch model, its weights,
    # and the fitted thermometer (needed to binarize new data the same way)
    torch.save({'model': model, 'state_dict': model.state_dict(), 'thermometer': therm, 'config': cfg,
                'n_inputs': xb_train.shape[1], 'run': RUN_ID}, f'{OUT}/{tag}_model.pt')
    print(f'[{tag}] saved {OUT}/{tag}_model.pt')

    r = hardware_report(model, xb_test, y_test, tag, input_reg=INPUT_REG, fast=FAST_RTL)
    r['paper_acc'], r['paper_LUTs'], r['paper_FFs'] = cfg['paper']
    results.append(r)
    models[tag] = model
    pd.DataFrame(results).to_csv(f'{OUT}/results.csv', index=False)


===== mnist_lg =====
Epoch 1/100, Train Loss: 0.1638, Train Accuracy: 0.9181, Test Accuracy: 0.9617 (17s)
Epoch 2/100, Train Loss: 0.0705, Train Accuracy: 0.9708, Test Accuracy: 0.9714 (17s)
Epoch 3/100, Train Loss: 0.0746, Train Accuracy: 0.9824, Test Accuracy: 0.9760 (18s)
Epoch 4/100, Train Loss: 0.0714, Train Accuracy: 0.9884, Test Accuracy: 0.9769 (17s)
Epoch 5/100, Train Loss: 0.0592, Train Accuracy: 0.9926, Test Accuracy: 0.9773 (18s)
Epoch 6/100, Train Loss: 0.0425, Train Accuracy: 0.9951, Test Accuracy: 0.9796 (17s)
Epoch 7/100, Train Loss: 0.0506, Train Accuracy: 0.9966, Test Accuracy: 0.9797 (18s)
Epoch 8/100, Train Loss: 0.0300, Train Accuracy: 0.9980, Test Accuracy: 0.9792 (18s)
Epoch 9/100, Train Loss: 0.0178, Train Accuracy: 0.9986, Test Accuracy: 0.9791 (18s)
Epoch 10/100, Train Loss: 0.0204, Train Accuracy: 0.9989, Test Accuracy: 0.9811 (18s)
Epoch 11/100, Train Loss: 0.0281, Train Accuracy: 0.9994, Test Accuracy: 0.9803 (18s)
Epoch 12/100, Train Loss: 0.0224, Train A

In [ ]:
def load_dwn(tag, run=None):
    path = f'{BASE}/run_{run if run is not None else RUN_ID}/{tag}_model.pt'
    ck = torch.load(path, map_location=device, weights_only=False)   # weights_only=False: the file also holds the thermometer object
    model = ck['model'].to(device).eval()
    return model, ck['thermometer'], ck

## Vivado: synthesize the same Verilog and compare

The cell below runs Vivado in batch mode on every `.v` file in `OUT`: out-of-context synthesis (the designs have more input bits
than the chip has pins), a 4 ns clock, the synthesis and implementation strategies chosen in the Options cell, then place & route. It reads **Slice LUTs** and **Slice Registers** from `report_utilization` and the
worst slack from timing, and puts them next to the paper's numbers. Reports and logs for each model go to `OUT/vivado/<model>/`.

You can run it on its own after a restart: run the Setup and import cells first, then this section. It reads `results.csv`, so
the models don't need retraining.

- **The paper's Table 2 used a Virtex UltraScale+ `xcvu9p-flgb2104-2-i`** with Vivado's `Flow_PerfOptimized_high` strategy and the highest clock it could reach.
  The Zynq Z-7045 is only used for Table 1. This Vivado install has no UltraScale+ (or Zynq/Kintex-7) parts, so the cell uses an Artix-7.
  LUT/FF counts are roughly comparable; Fmax is not.
- `VIVADO_PATH` points at this machine's Vivado 2026.1. Set it to `None` to search `C:\Xilinx`, `C:\AMDDesignTools`, etc.
- Place & route of `mnist_lg` / `jsc_lg` takes much longer than the small models. `VIVADO_IMPL = False` does synthesis only, which is much faster.

In [ ]:
for r in results:
    print(f"{OUT}/{r['model']}.v")

c:/Users/youse/Downloads/DWN/run_1/mnist_lg.v


In [ ]:
import glob, csv
# The paper's Table 2 used xcvu9p-flgb2104-2-i (Virtex UltraScale+, Vivado Flow_PerfOptimized_high); no UltraScale+ parts are installed here.
# Artix-7 also has LUT6 + FFs, so LUT/FF counts are roughly comparable, but adders use CARRY4 (vs CARRY8) and Fmax will be much lower.
VIVADO_SRC = VIVADO_SRC or OUT

# What each strategy passes to synth_design (from the Vivado strategy definitions)
SYNTH_STRATEGIES = {
    'Vivado Synthesis Defaults': '-flatten_hierarchy rebuilt',
    'Flow_PerfOptimized_high':   '-flatten_hierarchy rebuilt -fanout_limit 400 -fsm_extraction one_hot -keep_equivalent_registers -resource_sharing off -no_lc -shreg_min_size 5',
    'Flow_AreaOptimized_high':   '-flatten_hierarchy rebuilt -directive AreaOptimized_high',
    'Flow_AreaOptimized_medium': '-flatten_hierarchy rebuilt -directive AreaOptimized_medium',
    'Flow_RuntimeOptimized':     '-directive RuntimeOptimized -flatten_hierarchy none -fsm_extraction off',
}
# What each implementation strategy passes to opt / place / phys_opt / route
IMPL_STRATEGIES = {
    'Vivado Implementation Defaults': dict(opt='', place='', phys_opt=None, route=''),
    'Performance_Explore': dict(opt='-directive Explore', place='-directive Explore', phys_opt='-directive Explore', route='-directive Explore'),
}
if VIVADO_SYNTH_STRATEGY not in SYNTH_STRATEGIES:
    raise ValueError(f'VIVADO_SYNTH_STRATEGY must be one of {list(SYNTH_STRATEGIES)}')
if VIVADO_IMPL_STRATEGY not in IMPL_STRATEGIES:
    raise ValueError(f'VIVADO_IMPL_STRATEGY must be one of {list(IMPL_STRATEGIES)}')
VIVADO_TCL = """\
read_verilog {{{v}}}
read_xdc -mode out_of_context {{{out}/clock.xdc}}
synth_design -top dwn_top -part {part} -mode out_of_context {synth_opts}
report_utilization -file {{{out}/synth_util.txt}}
report_timing_summary -file {{{out}/synth_timing.txt}}
set stage synth
if {{{impl}}} {{
    opt_design {opt}
    place_design {place}
{phys_opt}    route_design {route}
    report_utilization -file {{{out}/impl_util.txt}}
    report_timing_summary -file {{{out}/impl_timing.txt}}
    set stage impl
}}
set p [get_timing_paths -max_paths 1 -nworst 1 -setup]
set wns nan
if {{[llength $p]}} {{ set wns [get_property SLACK $p] }}
set f [open {{{out}/wns.txt}} w]
puts $f "$stage $wns"
close $f
"""

def find_vivado():
    if VIVADO_PATH and os.path.exists(VIVADO_PATH):
        return VIVADO_PATH
    if shutil.which('vivado'):
        return shutil.which('vivado')
    hits = sorted(h for d in ['C:', 'D:'] for p in ['Xilinx/Vivado/*/bin', 'Xilinx/*/Vivado/bin', 'AMDDesignTools/*/Vivado/bin', '*/Vivado/bin']
                  for h in glob.glob(f'{d}/{p}/vivado.bat'))
    return hits[-1] if hits else None   # newest version

def parse_util(path):
    # "| Slice LUTs*  |  4536 |  0 |  0 | 218600 |  2.08 |" (7-series) or "| CLB LUTs* |" (UltraScale)
    luts = ffs = None
    if os.path.exists(path):
        for line in open(path):
            m = re.match(r'^\|\s*(Slice|CLB) LUTs\*?\s*\|\s*(\d+)', line)
            if m and luts is None:
                luts = int(m.group(2))
            m = re.match(r'^\|\s*(Slice|CLB) Registers\s*\|\s*(\d+)', line)
            if m and ffs is None:
                ffs = int(m.group(2))
    return luts, ffs

def vivado_synth(vivado, model):
    out = f'{VIVADO_SRC}/vivado/{model}'
    os.makedirs(out, exist_ok=True)
    for f in glob.glob(f'{out}/*.txt'):   # drop reports from an earlier run
        os.remove(f)
    # the clock goes in an XDC read before synthesis, so the performance strategies can optimize for it
    open(f'{out}/clock.xdc', 'w').write(f'create_clock -name clk -period {VIVADO_PERIOD} [get_ports clk]\n')
    st = IMPL_STRATEGIES[VIVADO_IMPL_STRATEGY]
    open(f'{out}/run.tcl', 'w').write(VIVADO_TCL.format(v=f'{VIVADO_SRC}/{model}.v', part=VIVADO_PART, out=out, impl=int(VIVADO_IMPL),
                                                        synth_opts=SYNTH_STRATEGIES[VIVADO_SYNTH_STRATEGY],
                                                        opt=st['opt'], place=st['place'], route=st['route'],
                                                        phys_opt=f"    phys_opt_design {st['phys_opt']}\n" if st['phys_opt'] is not None else ''))
    t0 = time.time()
    print(f'[{model}] Vivado {"synth + place & route" if VIVADO_IMPL else "synth"} ...', end=' ', flush=True)
    rc = subprocess.run([vivado, '-mode', 'batch', '-nojournal', '-log', f'{out}/vivado.log', '-source', f'{out}/run.tcl'],
                        cwd=out, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, shell=vivado.lower().endswith('.bat')).returncode
    if rc != 0:
        print(f'FAILED (exit {rc}), see {out}/vivado.log')
        for e in [l.strip() for l in open(f'{out}/vivado.log', errors='replace') if l.startswith('ERROR')][:5]:
            print('   ', e)
    else:
        print(f'{(time.time() - t0) / 60:.1f} min')

def vivado_row(model, res):
    out = f'{VIVADO_SRC}/vivado/{model}'
    s_luts, s_ffs = parse_util(f'{out}/synth_util.txt')
    i_luts, i_ffs = parse_util(f'{out}/impl_util.txt')
    stage, wns = open(f'{out}/wns.txt').read().split() if os.path.exists(f'{out}/wns.txt') else (None, None)
    wns = float(wns) if wns is not None else None
    y = res.get(model, {})
    return {'model': model, 'acc %': round(float(y['acc_sim']) * 100, 2) if 'acc_sim' in y else None,
            'paper_acc': y.get('paper_acc'),
            'vivado_LUTs': i_luts if i_luts is not None else s_luts, 'paper_LUTs': y.get('paper_LUTs'),
            'vivado_FFs': i_ffs if i_ffs is not None else s_ffs, 'paper_FFs': y.get('paper_FFs'),
            'vivado_stage': stage, 'synth_strategy': VIVADO_SYNTH_STRATEGY, 'impl_strategy': VIVADO_IMPL_STRATEGY, 'synth_LUTs': s_luts, 'synth_FFs': s_ffs, 'WNS_ns': wns,
            'Fmax_MHz': round(1000 / (VIVADO_PERIOD - wns), 1) if wns is not None and wns == wns else None}

vivado = find_vivado()
models = VIVADO_MODELS or sorted(os.path.basename(p)[:-2] for p in glob.glob(f'{VIVADO_SRC}/*.v'))
if vivado is None:
    print('Vivado not found: install it or set VIVADO_PATH. Showing results from any earlier Vivado runs.')
else:
    print('Vivado:', vivado, '| part:', VIVADO_PART, '| clock:', VIVADO_PERIOD, 'ns', '| synth:', VIVADO_SYNTH_STRATEGY, '| impl:', VIVADO_IMPL_STRATEGY)
    for model in models:
        vivado_synth(vivado, model)

res = {r['model']: r for r in csv.DictReader(open(f'{VIVADO_SRC}/results.csv'))} if os.path.exists(f'{VIVADO_SRC}/results.csv') else {}
vdf = pd.DataFrame([vivado_row(m, res) for m in models])
vdf.to_csv(f'{VIVADO_SRC}/vivado/vivado_results.csv', index=False) if os.path.isdir(f'{VIVADO_SRC}/vivado') else None
vdf[['model', 'acc %', 'paper_acc', 'vivado_LUTs', 'paper_LUTs', 'vivado_FFs', 'paper_FFs', 'vivado_stage', 'WNS_ns', 'Fmax_MHz']]

Vivado: D:/2026.1/Vivado/bin/vivado.bat | part: xc7a200tffg1156-2 | clock: 4.0 ns | synth: Flow_PerfOptimized_high | impl: Vivado Implementation Defaults
[mnist_lg] Vivado synth + place & route ... 6.6 min
[mnist_sm] Vivado synth + place & route ... 4.2 min


,model,acc %,paper_acc,vivado_LUTs,paper_LUTs,vivado_FFs,paper_FFs,vivado_stage,WNS_ns,Fmax_MHz
0,mnist_lg,98.29,98.3,4507,4082,3358,3385,impl,0.054,253.4
1,mnist_sm,NaN,None,2240,None,1738,None,impl,0.165,260.8
